# MBA Friction Hunter
**Find where AI creates real learning value in an MBA curriculum — then build a working prototype.**

This notebook runs a 6-stage analysis pipeline on any MBA syllabus and outputs:
- A friction map of where students get stuck
- 2–4 prioritized AI use cases
- Architecture recommendations grounded in current research
- A design brief with realistic dialogue examples
- A working Claude system prompt ready to deploy

---

## Step 1 — Install & configure

In [ ]:
!pip install -q anthropic
!git clone -q https://github.com/johnschwab1/MBA_friction_hunter.git
import os, sys
os.chdir('MBA_friction_hunter')
print('Ready.')

In [ ]:
# Loads your Anthropic API key from Colab Secrets (key icon in left sidebar)
# Secret name: Claude_A
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('Claude_A')
print('API key loaded.')

## Step 2 — Load shared utilities

In [ ]:
import re, anthropic
from pathlib import Path
from IPython.display import Markdown, display

MODEL = 'claude-opus-4-8'
client = anthropic.Anthropic()

def extract_system_prompt(md_path):
    text = Path(md_path).read_text()
    match = re.search(r'## System Prompt\s*\n(.*)', text, re.DOTALL)
    if not match:
        raise ValueError(f'No System Prompt section in {md_path}')
    return match.group(1).strip()

def call_claude(system, user, use_search=False, max_tokens=8192):
    tools = [{'type': 'web_search_20250305', 'name': 'web_search', 'max_uses': 5}] if use_search else []
    messages = [{'role': 'user', 'content': user}]
    for _ in range(20):
        kwargs = dict(model=MODEL, max_tokens=max_tokens, system=system, messages=messages)
        if tools: kwargs['tools'] = tools
        resp = client.messages.create(**kwargs)
        text_parts, tool_uses = [], []
        for block in resp.content:
            if hasattr(block, 'text'): text_parts.append(block.text)
            if getattr(block, 'type', '') == 'tool_use': tool_uses.append(block)
        if resp.stop_reason == 'end_turn' or not tool_uses:
            return '\n'.join(text_parts)
        messages.append({'role': 'assistant', 'content': resp.content})
        messages.append({'role': 'user', 'content': [
            {'type': 'tool_result', 'tool_use_id': tu.id, 'content': ''} for tu in tool_uses
        ]})
    return '\n'.join(text_parts)

outputs = {}
criteria_text = Path('stage_1_criteria_layer.md').read_text()
print('Utilities loaded.')

## Step 3 — Provide the syllabus

**Option A:** Search for one by name (uses web search)  
**Option B:** Paste the text directly into `SYLLABUS_TEXT` below

In [ ]:
# Option A: search for a syllabus
SEARCH_QUERY = 'Haas MBA marketing syllabus 2025'  # <-- edit this

search_system = (
    'You are a research assistant. Find the requested MBA course syllabus, '
    'retrieve its full content, and return the complete text including course description, '
    'learning objectives, weekly schedule, assignments, and readings. '
    'Return only the syllabus content itself.'
)

print(f'Searching for: {SEARCH_QUERY}...')
syllabus = call_claude(search_system, f'Find and return the full text of: {SEARCH_QUERY}', use_search=True, max_tokens=4096)
print(f'Fetched {len(syllabus):,} chars.')
display(Markdown(syllabus[:2000] + ('...' if len(syllabus) > 2000 else '')))

In [ ]:
# Option B: paste syllabus text directly (skip if you ran Option A above)
SYLLABUS_TEXT = '''
Paste your syllabus here...
'''

# Uncomment the next line to use this instead of the search result above:
# syllabus = SYLLABUS_TEXT.strip()

---
## Stage 2 — Friction Hunter
*Classifies where students get stuck and scores each friction type against our criteria.*

In [ ]:
print('Running Stage 2: Friction Hunter...')
system2 = extract_system_prompt('stage_2_friction_hunter.md')
user2 = f'CRITERIA REFERENCE (use to calibrate your scoring):\n\n{criteria_text}\n\n---\n\nCURRICULUM TO ANALYZE:\n\n{syllabus}'
outputs[2] = call_claude(system2, user2)
print('Done.')
display(Markdown(outputs[2]))

---
## Stage 3 — Use Case Selector
*Picks 2–4 friction points worth building into AI tools, ranked by learning impact and feasibility.*

In [ ]:
print('Running Stage 3: Use Case Selector...')
system3 = extract_system_prompt('stage_3_use_case_selector.md')
outputs[3] = call_claude(system3, outputs[2])
print('Done.')
display(Markdown(outputs[3]))

---
## Stage 4 — Architecture Hunter
*Searches current research (GitHub, papers, Stanford SCALE, MIT RAISE) to find the right AI architecture for each use case.*

> Web search is active for this stage.

In [ ]:
print('Running Stage 4: Architecture Hunter (web search active)...')
system4 = extract_system_prompt('stage_4_architecture_hunter.md')
outputs[4] = call_claude(system4, outputs[3], use_search=True)
print('Done.')
display(Markdown(outputs[4]))

---
## Stage 5 — Design Brief
*One-page brief per use case: what the friction is, what the AI does, a realistic dialogue, and success/failure signals.*

In [ ]:
print('Running Stage 5: Design Brief...')
system5 = extract_system_prompt('stage_5_design_brief_generator.md')
user5 = (
    f'PRIORITIZED USE CASES (Stage 3):\n\n{outputs[3]}\n\n---\n\n'
    f'ARCHITECTURE RECOMMENDATIONS (Stage 4):\n\n{outputs[4]}'
)
outputs[5] = call_claude(system5, user5)
print('Done.')
display(Markdown(outputs[5]))

---
## Stage 6 — Prototype
*A working Claude system prompt you can paste into a Claude Project and demo on the spot.*

In [ ]:
print('Running Stage 6: Prototype Generator...')
system6 = extract_system_prompt('stage_6_prototype_generator.md')
outputs[6] = call_claude(system6, outputs[5])
print('Done.')
display(Markdown(outputs[6]))

---
## Save all outputs

In [ ]:
from pathlib import Path
from datetime import datetime

stage_names = {2: 'friction_hunter', 3: 'use_case_selector', 4: 'architecture_hunter', 5: 'design_brief', 6: 'prototype'}
run_name = datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = Path(f'output/{run_name}')
out_dir.mkdir(parents=True, exist_ok=True)

for stage_num, content in outputs.items():
    path = out_dir / f'stage_{stage_num:02d}_{stage_names[stage_num]}.md'
    path.write_text(content)
    print(f'Saved {path}')

print(f'\nAll outputs saved to {out_dir}')